# Lab 15: Integration Demo — คณิตศาสตร์ทั้ง 4 CLO ใน 1 Dataset
> Week 15 | CLO1 + CLO2 + CLO3 + CLO4 | Final Integration

---

สัปดาห์สุดท้ายนี้เราจะรวมทุกอย่างที่เรียนมาตลอด 15 สัปดาห์เข้าด้วยกัน โดยใช้ **California Housing Dataset** เป็น dataset เดียวที่แสดงให้เห็นว่าคณิตศาสตร์ทั้ง 4 CLO เชื่อมกันอย่างไรในการวิเคราะห์ข้อมูลจริง เป้าหมายคือให้เห็น 'ภาพรวม' ของ Data Science pipeline ตั้งแต่ linear algebra ไปจนถึง model selection และเพื่อเป็น reference สำหรับ Final Project

**Dataset:** California Housing (sklearn) — 20,640 ย่านที่อยู่อาศัยในรัฐ California ปี 1990

| CLO | หัวข้อ | Part ใน Notebook |
|-----|-------|----------------|
| CLO1 | Linear Algebra: Feature matrix, Covariance, PCA | Part 1 |
| CLO2 | Statistical Learning: EDA, Bias-Variance | Part 2 |
| CLO3 | Regression + Classification | Part 3 |
| CLO4 | Model Selection: Cross-Validation | Part 4 |

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
# วัตถุประสงค์: โหลด libraries ทั้งหมดที่ใช้ใน integration demo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, r2_score

# ─── ตั้งค่า global settings ──────────────────────────────────────
# วัตถุประสงค์: ให้ผลลัพธ์ reproducible และ plot สวยงาม
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('All libraries loaded. Dataset: California Housing')

---
## Part 1: CLO1 — Linear Algebra
### Feature Matrix, Covariance Matrix, PCA

ในส่วนนี้เราจะใช้ Linear Algebra (Week 1–4) เพื่อ:
1. สร้าง **feature matrix** X และดูโครงสร้าง
2. คำนวณ **covariance matrix** C = (1/n-1) XᵀX ด้วย hand-coded computation
3. ทำ **PCA** เพื่อลด dimension จาก 8 → 2 และ visualize structure ของ data

เชื่อมกับ Week 4: นี่คือ `np.linalg.eigh(C)` + manual sort + projection ที่ทำใน Lab 04

In [ ]:
# ─── โหลด California Housing Dataset ─────────────────────────────
# วัตถุประสงค์: ข้อมูล 20,640 ย่านใน California ใช้เป็น single dataset ตลอด lab
housing = fetch_california_housing()
X = housing.data.astype(float)
y = housing.target
feature_names = housing.feature_names

print('=== Feature Matrix X ===')
print(f'Shape: {X.shape}  ({X.shape[0]} observations x {X.shape[1]} features)')
print(f'Features: {feature_names}')
print(f'Target: MedHouseVal (median house value, in $100K)')
print(f'Target range: ${y.min():.2f} – ${y.max():.2f} x100K')

In [ ]:
# ─── PCA by Hand (CLO1 connection) ────────────────────────────────
# วัตถุประสงค์: แสดงว่า PCA จาก Week 4 apply กับ real dataset ได้จริง

# Step 1: Center data (subtract mean)
X_mean    = X.mean(axis=0)
X_centered = X - X_mean

# Step 2: Covariance matrix C = (1/(n-1)) * X_centered^T @ X_centered
n = X.shape[0]
C = (1 / (n - 1)) * X_centered.T @ X_centered
print(f'Covariance matrix C: shape = {C.shape} (8x8)')

# Step 3: Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(C)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Step 4: Explained variance
explained_ratio = eigenvalues / eigenvalues.sum()
cumulative_ratio = np.cumsum(explained_ratio)

# Step 5: Project to 2D
X_pca = X_centered @ eigenvectors[:, :2]

# ─── Plot: Scree + PCA scatter ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, 9), explained_ratio * 100, color='steelblue', alpha=0.7)
axes[0].plot(range(1, 9), cumulative_ratio * 100, 'r-o', lw=2, label='Cumulative')
axes[0].axhline(y=95, color='gray', linestyle='--', label='95% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('CLO1: Scree Plot — California Housing')
axes[0].legend()

scatter = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                          c=y, cmap='viridis', alpha=0.2, s=1)
plt.colorbar(scatter, ax=axes[1], label='House Value ($100K)')
axes[1].set_xlabel(f'PC1 ({explained_ratio[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({explained_ratio[1]*100:.1f}% variance)')
axes[1].set_title('CLO1: PCA 2D Projection — colour = house value')

plt.tight_layout()
plt.show()

print('PC1 loadings (feature contributions):')
for name, load in zip(feature_names, eigenvectors[:, 0]):
    print(f'  {name:12s}: {load:+.4f}')

---
## Part 2: CLO2 — Statistical Learning
### EDA + Bias-Variance Trade-Off

ในส่วนนี้เราจะใช้ Statistical Learning (Week 5–6) เพื่อ:
1. **EDA** — distribution, correlation heatmap, scatter matrix
2. **Bias-Variance** — แสดง U-curve ของ polynomial regression บน 1 feature

เชื่อมกับ Week 5–6: Y = f(X) + ε, Training vs Test MSE, Bias² + Variance + Irreducible

In [ ]:
# ─── EDA: Correlation Heatmap ─────────────────────────────────────
# วัตถุประสงค์: ดูว่า feature ไหน correlate กับ target (house value) มากที่สุด
df = pd.DataFrame(X, columns=feature_names)
df['MedHouseVal'] = y

corr = df.corr()
corr_with_target = corr['MedHouseVal'].drop('MedHouseVal').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation with target
colors = ['coral' if v < 0 else 'steelblue' for v in corr_with_target]
axes[0].barh(corr_with_target.index, corr_with_target.values, color=colors)
axes[0].axvline(x=0, color='black', lw=0.8)
axes[0].set_xlabel('Pearson Correlation with MedHouseVal')
axes[0].set_title('CLO2: Feature Correlations with Target')

# Scatter: MedInc vs MedHouseVal (highest correlation)
axes[1].scatter(df['MedInc'], df['MedHouseVal'], alpha=0.05, s=1, color='steelblue')
axes[1].set_xlabel('Median Income (x$10K)')
axes[1].set_ylabel('Median House Value ($100K)')
axes[1].set_title(f'CLO2: MedInc vs MedHouseVal  (r={corr_with_target["MedInc"]:.3f})')

plt.tight_layout()
plt.show()

print('Top 3 features correlated with house value:')
print(corr_with_target.tail(3))

In [ ]:
# ─── Bias-Variance: U-curve บน MedInc → MedHouseVal ──────────────
# วัตถุประสงค์: แสดง U-curve ของ Test MSE เหมือน Lab 06
X_single = X[:, 0:1]  # MedInc feature เดียว
X_tr, X_te, y_tr, y_te = train_test_split(X_single, y, test_size=0.2, random_state=42)

degrees = range(1, 11)
train_mses, test_mses = [], []

for deg in degrees:
    m = make_pipeline(PolynomialFeatures(deg), Ridge(alpha=0.01))
    m.fit(X_tr, y_tr)
    train_mses.append(mean_squared_error(y_tr, m.predict(X_tr)))
    test_mses.append(mean_squared_error(y_te,  m.predict(X_te)))

plt.figure(figsize=(8, 4))
plt.plot(degrees, train_mses, 'b-o', label='Train MSE')
plt.plot(degrees, test_mses,  'r-o', label='Test MSE')
plt.xlabel('Polynomial Degree (Flexibility →)')
plt.ylabel('MSE')
plt.title('CLO2: Bias-Variance U-Curve — MedInc → MedHouseVal')
plt.legend()
plt.tight_layout()
plt.show()

opt = list(degrees)[test_mses.index(min(test_mses))]
print(f'Optimal degree (from single split): {opt}')
print(f'Min Test MSE: {min(test_mses):.4f}  |  RMSE ≈ ${np.sqrt(min(test_mses))*100:.0f}')

---
## Part 3: CLO3 — Regression และ Classification
### Multiple Linear Regression + Logistic Regression

ในส่วนนี้เราจะ:
1. **Multiple Linear Regression** — predict ราคาบ้านจาก 8 features (Week 9)
2. **Logistic Regression** — classify ว่าย่านนั้น 'แพง' หรือ 'ถูก' (above/below median) (Week 11)

เชื่อมกับ Week 9 (MLR), Week 11 (Logistic)

In [ ]:
# ─── Multiple Linear Regression (CLO3) ────────────────────────────
# วัตถุประสงค์: predict house value จากทุก 8 features พร้อมกัน
X_tr_f, X_te_f, y_tr_f, y_te_f = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_tr_f, y_tr_f)
y_pred_lr = lr.predict(X_te_f)

mse_lr = mean_squared_error(y_te_f, y_pred_lr)
r2_lr  = r2_score(y_te_f, y_pred_lr)

print('=== Multiple Linear Regression ===')
print(f'Test MSE: {mse_lr:.4f}  |  RMSE ≈ ${np.sqrt(mse_lr)*100:.0f}')
print(f'Test R²:  {r2_lr:.4f}  ({r2_lr*100:.1f}% variance explained)')
print()
print('Coefficients (β̂):')
for name, coef in zip(feature_names, lr.coef_):
    print(f'  {name:12s}: {coef:+.4f}')

# ─── Plot: Actual vs Predicted ────────────────────────────────────
plt.figure(figsize=(7, 5))
plt.scatter(y_te_f, y_pred_lr, alpha=0.1, s=2, color='steelblue')
plt.plot([0, 5], [0, 5], 'r--', lw=1.5, label='Perfect prediction')
plt.xlabel('Actual House Value ($100K)')
plt.ylabel('Predicted House Value ($100K)')
plt.title(f'CLO3: Multiple Linear Regression  R²={r2_lr:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ─── Logistic Regression (CLO3) ────────────────────────────────────
# วัตถุประสงค์: จัด classify ย่านเป็น 'แพง' (1) หรือ 'ถูก' (0) เทียบกับ median
y_binary = (y > np.median(y)).astype(int)

X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(X, y_binary, test_size=0.2, random_state=42)

# StandardScaler สำคัญสำหรับ Logistic Regression
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_b)
X_te_s = scaler.transform(X_te_b)

logit = LogisticRegression(max_iter=1000, random_state=42)
logit.fit(X_tr_s, y_tr_b)
acc = accuracy_score(y_te_b, logit.predict(X_te_s))

print('=== Logistic Regression ===')
print(f'Task: classify ราคาบ้าน > median ({np.median(y):.2f} x$100K)?')
print(f'Test Accuracy: {acc:.4f}  ({acc*100:.1f}%)')
print()
print('Coefficients (log-odds effect):')
for name, coef in zip(feature_names, logit.coef_[0]):
    direction = 'more likely EXPENSIVE' if coef > 0 else 'more likely cheap'
    print(f'  {name:12s}: {coef:+.4f}  → {direction}')

---
## Part 4: CLO4 — Model Selection
### Cross-Validation เปรียบเทียบหลาย Models

ในส่วนนี้เราจะใช้ **5-Fold Cross-Validation** เพื่อเปรียบเทียบ 5 models อย่างยุติธรรม เชื่อมกับ Week 14 (Cross-Validation) — แทนที่จะ evaluate บน single test split เราใช้ 5 folds เพื่อได้ estimate ที่น่าเชื่อถือกว่า

In [ ]:
# ─── Cross-Validation: เปรียบเทียบ 5 Models (CLO4) ─────────────────
# วัตถุประสงค์: หา model ที่ดีที่สุดอย่างเป็นธรรมโดยใช้ 5-fold CV
scaler_cv = StandardScaler()
X_scaled  = scaler_cv.fit_transform(X)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)':   Ridge(alpha=1.0),
    'Ridge (alpha=10)':  Ridge(alpha=10.0),
    'KNN k=5':           KNeighborsRegressor(n_neighbors=5),
    'KNN k=20':          KNeighborsRegressor(n_neighbors=20),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}
print(f'{"Model":<22} | {"CV MSE (mean)":>13} | {"CV MSE (std)":>12} | {"CV RMSE":>8}')
print('-' * 65)

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=kf, scoring='neg_mean_squared_error')
    cv_mse  = -scores.mean()
    cv_std  = scores.std()
    cv_results[name] = (cv_mse, cv_std)
    print(f'{name:<22} | {cv_mse:>13.4f} | {cv_std:>12.4f} | {np.sqrt(cv_mse):>8.4f}')

# ─── Plot CV Results ──────────────────────────────────────────────
model_names = list(cv_results.keys())
means = [cv_results[m][0] for m in model_names]
stds  = [cv_results[m][1] for m in model_names]

plt.figure(figsize=(10, 5))
colors_bar = ['steelblue' if 'Ridge' in m or 'Linear' in m else 'coral' for m in model_names]
bars = plt.bar(model_names, means, yerr=stds, capsize=5, color=colors_bar, alpha=0.8)
plt.ylabel('CV MSE (lower is better)')
plt.title('CLO4: 5-Fold Cross-Validation — Model Comparison')
plt.xticks(rotation=15, ha='right')
best_idx = means.index(min(means))
bars[best_idx].set_edgecolor('gold')
bars[best_idx].set_linewidth(3)
plt.tight_layout()
plt.show()

best_model = model_names[best_idx]
print(f'Best model (by CV MSE): {best_model}')
print(f'CV RMSE ≈ ${np.sqrt(means[best_idx])*100:.0f}')

---
## สรุป: Integration Map

```
California Housing Dataset (20,640 observations, 8 features)
          │
          ├── CLO1: Feature Matrix X (20640×8)
          │         Covariance Matrix C (8×8)  ← Matrix multiplication
          │         PCA → 2 components = 98% variance
          │
          ├── CLO2: EDA → MedInc มี correlation สูงสุดกับ target
          │         Bias-Variance U-curve → optimal degree ≈ 3
          │
          ├── CLO3: Multiple Linear Regression → R² ≈ 0.60
          │         Logistic Regression → Accuracy ≈ 80%
          │
          └── CLO4: 5-Fold CV → Ridge (alpha=1) wins over plain Linear Reg
                    Cross-Validation = unbiased estimate of generalization
```

**สิ่งที่เห็นจาก integration นี้:**
- **CLO1** ให้ insight ว่า PC1 dominated โดย latitude/longitude (geographic clustering)
- **CLO2** ยืนยันว่า MedInc เป็น best single predictor
- **CLO3** แสดงว่า linear model explain ได้ 60% ของ variance — ยังมีอีก 40% ที่ต้องการ nonlinear model
- **CLO4** แสดงว่า Ridge regularization ช่วยได้เล็กน้อย และ KNN ยังไม่ดีกว่า linear สำหรับ data ขนาดนี้

**สำหรับ Final Project:** ทำในลักษณะเดียวกันนี้ แต่กับ dataset ที่กลุ่มเลือกเอง